# Choix du DATASET



# Classification de recettes par type de cuisine sans apprentissage

Pour chaque cuisine on construit un vecteur de poids (basé sur les ingrédients les plus fréquents et pondérés par un IDF). Une recette est ensuite comparée à chaque vecteur par un produit scalaire, puis un softmax donne une probabilité par cuisine.

Importation des données et chargement des librairies

In [ ]:

import math
import json
import numpy as np
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


with open("train.json", "r", encoding="utf-8") as f:
    data = json.load(f)


Exploration du dataset

Récupération d'informations sur le dataset

In [ ]:
nb_data = len(data)
classes = [recipe["cuisine"] for recipe in data]
classe_counts = Counter(classes)
nb_classes = len(classe_counts)

print(f"Nombre total de données : {nb_data}")
print(f"Nombre de classes : {nb_classes}")

print("\nNombre de données par classe :")

for cuisine, count in classe_counts.most_common():
    pourcentage = (count / nb_data) * 100
    print(f"{cuisine} : {count} ({pourcentage:.2f}%)")



On compte le nombre total de recettes, le nombre de cuisines (classes) et la répartition par classe.
Observation Le dataset est déséquilibré : l'italien et le mexicain représentent à eux seuls plus d'un tiers des recettes, alors que les cuisines brésilienne ou russe en représentent environ 1 %.

Séparation en train / validation / test

On découpe le dataset en trois parties :
- train (70 %)
- validation (15 %)
- test (15 %)

In [ ]:
# fait par chat gpt
SEED = 42
RATIO_TRAIN, RATIO_VAL = 0.70, 0.15   # le reste (0.15) va au test

rng = np.random.default_rng(SEED)

# Indices des recettes regroupés par cuisine
indices_par_classe = defaultdict(list)
for i, recipe in enumerate(data):
    indices_par_classe[recipe["cuisine"]].append(i)

train_idx, val_idx, test_idx = [], [], []
for cuisine, idx in indices_par_classe.items():
    idx = np.array(idx)
    rng.shuffle(idx)
    n_train = int(RATIO_TRAIN * len(idx))
    n_val = int(RATIO_VAL * len(idx))
    train_idx += idx[:n_train].tolist()
    val_idx += idx[n_train:n_train + n_val].tolist()
    test_idx += idx[n_train + n_val:].tolist()

train_data = [data[i] for i in train_idx]
val_data = [data[i] for i in val_idx]
test_data = [data[i] for i in test_idx]

print(f"Train      : {len(train_data)} recettes")
print(f"Validation : {len(val_data)} recettes")
print(f"Test       : {len(test_data)} recettes")

# Vérification de la stratification sur les 3 classes les plus fréquentes
for nom, subset in [("train", train_data), ("val", val_data), ("test", test_data)]:
    counts = Counter(r["cuisine"] for r in subset)
    print(nom, {c: f"{counts[c] / len(subset) * 100:.1f}%" for c, _ in classe_counts.most_common(3)})

Construction du modèle 

Comptage des ingrédients par cuisine

Pour chaque cuisine, on compte dans combien de recettes chaque ingrédient apparaît.

In [ ]:
ingredients_par_cuisine = defaultdict(Counter)

for recipe in train_data:
    cuisine = recipe["cuisine"]

    # set permet de mettre l'ingredient que une seul fois dans le dictionnaire
    ingredients = set(recipe["ingredients"])

    for ingredient in ingredients:
        ingredients_par_cuisine[cuisine][ingredient] += 1

Le `set` évite de compter deux fois un même ingrédient s'il est répété dans une recette. Le résultat est un dictionnaire `cuisine -> Counter(ingrédient -> nombre de recettes)`.

Les 30 ingrédients les plus fréquents par cuisine

In [ ]:
top_par_cuisine = {}

for cuisine, counter in ingredients_par_cuisine.items():
    top_par_cuisine[cuisine] = [
        ingredient
        for ingredient, count in counter.most_common(30)
    ]

print(top_par_cuisine)

On ne garde que les 30 ingrédients les plus fréquents de chaque cuisine.

Vocabulaire

In [ ]:
vocabulary = set()

for cuisine in top_par_cuisine:

    vocabulary.update(
        top_par_cuisine[cuisine]
    )

vocabulary = sorted(vocabulary)

mot_index = {
    ingredient: i
    for i, ingredient in enumerate(vocabulary)
}


print("Nombre de classes :", len(top_par_cuisine))
print("Taille du vocabulaire :", len(vocabulary))

Le vocabulaire est l'union des top-30 de toutes les cuisines. On le trie pour avoir un ordre fixe, puis on associe chaque ingrédient à un index (`mot_index`) : cet index sera la position de l'ingrédient dans les vecteurs. Un ingrédient absent du vocabulaire sera ignoré lors de la prédiction.

IDF

In [ ]:
df = Counter()
for cuisine in top_par_cuisine:
    for ingredient in top_par_cuisine[cuisine]:
        df[ingredient] += 1

idf = {}
for ingredient in vocabulary:
    idf[ingredient] = math.log(len(top_par_cuisine)/df[ingredient])


`df[ingrédient]` est le nombre de cuisines dont le top-30 contient cet ingrédient. L'IDF vaut `log(nombre de cuisines / df)` :
- un ingrédient présent dans **toutes** les cuisines a un IDF de 0 : il ne discrimine rien ;
- un ingrédient présent dans **peu** de cuisines a un IDF élevé

Vecteur de poids de chaque cuisine

In [ ]:
class_vectors = {}

for cuisine, top_ingredients in top_par_cuisine.items():
    vector = np.zeros(len(vocabulary))
    for ingredient in top_ingredients:
        index = mot_index[ingredient]
        # Fréquence de l'ingrédient dans cette classe
        frequency = ingredients_par_cuisine[cuisine][ingredient]
        vector[index] = idf[ingredient]
    class_vectors[cuisine] = vector

#print(class_vectors)

Pour chaque cuisine, on crée un vecteur de la taille du vocabulaire, rempli de 0. Pour chaque ingrédient de son top-30, on met son IDF à la position correspondante.

Remarque : la variable `frequency` est calculée mais n'est pas utilisée, le poids final ne dépend donc que de l'IDF. Pour pondérer aussi par la fréquence dans la cuisine (TF-IDF), il faudrait écrire `vector[index] =  idf[ingredient]`.

Fonctions de prédiction

In [ ]:
def recipe_to_vector(ingredients, word_to_index):
    
    # Vecteur rempli de 0
    vector = np.zeros(len(word_to_index))
    # Parcours des ingrédients de la recette
    for ingredient in ingredients:
        # On ignore les ingrédients qui ne sont pas dans le vocabulaire
        if ingredient in word_to_index:
            index = word_to_index[ingredient]
            vector[index] = 1

    return vector

def softmax(scores):
    # Stabilisation numérique
    scores = scores - np.max(scores)
    exp_scores = np.exp(scores)
    return exp_scores / np.sum(exp_scores)


def predict_recipe(recipe_vector, class_vectors):
    
    classes = list(class_vectors.keys())
    scores = []

    # Combinaison linéaire pour chaque classe
    for cuisine in classes:
        class_vector = class_vectors[cuisine]
        score = np.dot(class_vector, recipe_vector)
        scores.append(score)

    scores = np.array(scores)
    # Un seul softmax sur tous les scores
    probabilities = softmax(scores)
    # Classe ayant la probabilité maximale
    predicted_index = np.argmax(probabilities)
    predicted_class = classes[predicted_index]
    return predicted_class, probabilities, scores

Trois fonctions :
- `recipe_to_vector` transforme une recette en vecteur binaire (1 si l'ingrédient est présent, 0 sinon) ;
- `softmax` convertit des scores en probabilités qui somment à 1 (on soustrait le max pour éviter les dépassements numériques) ;
- `predict_recipe` calcule le produit scalaire entre la recette et le vecteur de chaque cuisine, applique le softmax et renvoie la cuisine de probabilité maximale.

Le score d'une cuisine est donc la somme des IDF des ingrédients de la recette présents dans le top-30 de cette cuisine.

Métriques d'évaluation

In [ ]:
# fait par chat gpt

def predict_dataset(dataset, class_vectors, word_to_index):
    """Renvoie (vraies classes, classes prédites) pour une liste de recettes."""
    y_true, y_pred = [], []
    for recipe in dataset:
        vector = recipe_to_vector(recipe["ingredients"], word_to_index)
        predicted_class, _, _ = predict_recipe(vector, class_vectors)
        y_true.append(recipe["cuisine"])
        y_pred.append(predicted_class)
    return y_true, y_pred


def accuracy(y_true, y_pred):
    """Proportion de recettes correctement classées."""
    return float(np.mean(np.array(y_true) == np.array(y_pred)))


def precision_par_classe(y_true, y_pred, classes):
    """Précision de chaque classe = VP / (VP + FP), et support (nb de vraies recettes)."""
    precisions, supports = {}, Counter(y_true)
    for c in classes:
        predits_c = sum(1 for p in y_pred if p == c)
        vp = sum(1 for t, p in zip(y_true, y_pred) if p == c and t == c)
        precisions[c] = vp / predits_c if predits_c > 0 else 0.0
    return precisions, supports


def rapport(nom, dataset, class_vectors, word_to_index, detail=True):
    y_true, y_pred = predict_dataset(dataset, class_vectors, word_to_index)
    classes_list = sorted(class_vectors.keys())
    prec, supports = precision_par_classe(y_true, y_pred, classes_list)

    acc = accuracy(y_true, y_pred)
    macro = float(np.mean(list(prec.values())))
    total = sum(supports.values())
    weighted = sum(prec[c] * supports[c] for c in classes_list) / total

    print(f"=== {nom} ({total} recettes) ===")
    print(f"Accuracy            : {acc:.4f}")
    print(f"Précision (macro)   : {macro:.4f}")
    print(f"Précision (pondérée): {weighted:.4f}")
    if detail:
        print("\nPrécision par classe :")
        for c in sorted(classes_list, key=lambda c: -prec[c]):
            print(f"  {c:<14} {prec[c]:.4f}  (support : {supports[c]})")
    return acc, macro, weighted

Les métriques utilisées :
- **Accuracy** : part des recettes dont la cuisine est correctement prédite ;
- **Précision d'une classe** : parmi les recettes prédites dans cette cuisine, part de celles qui en font réellement partie (`VP / (VP + FP)`) ;
- **Précision macro** : moyenne simple des précisions des 20 classes (toutes les cuisines pèsent pareil, donc sensible aux petites classes) ;
- **Précision pondérée** : moyenne pondérée par le nombre de recettes de chaque classe.

### Rappel, F1 et matrice de confusion

On ajoute trois métriques supplémentaires, calculées à partir de la **matrice de confusion** :
- **Rappel d'une classe** : parmi les recettes qui sont réellement de cette cuisine, part de celles que le modèle a retrouvées (`VP / (VP + FN)`) ;
- **F1 d'une classe** : moyenne harmonique de la précision et du rappel (`2 × précision × rappel / (précision + rappel)`). Elle est basse dès que l'une des deux est basse ;
- **Matrice de confusion** : tableau 20 × 20 où la ligne est la vraie cuisine et la colonne la cuisine prédite. La diagonale contient les recettes bien classées, le reste montre quelles cuisines sont confondues.

Comme pour la précision, on donne la moyenne **macro** (toutes les classes pèsent pareil) et la moyenne **pondérée** (pondérée par le nombre de recettes). Le rappel pondéré est toujours égal à l'accuracy.

In [ ]:
# chatgpt


classes_list = sorted(classe_counts.keys())


def matrice_confusion(y_true, y_pred, classes):
    """Matrice de confusion : lignes = vraie classe, colonnes = classe prédite."""
    index = {c: i for i, c in enumerate(classes)}
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[index[t], index[p]] += 1
    return cm


def metriques_depuis_matrice(cm):
    """Précision, rappel, F1 et support de chaque classe, calculés à partir de la matrice."""
    vp = np.diag(cm).astype(float)
    predits = cm.sum(axis=0)      # nb de recettes prédites dans chaque classe (VP + FP)
    supports = cm.sum(axis=1)     # nb de vraies recettes de chaque classe (VP + FN)

    precision = np.divide(vp, predits, out=np.zeros_like(vp), where=predits > 0)
    rappel = np.divide(vp, supports, out=np.zeros_like(vp), where=supports > 0)
    somme = precision + rappel
    f1 = np.divide(2 * precision * rappel, somme, out=np.zeros_like(vp), where=somme > 0)
    return precision, rappel, f1, supports


def resume_metriques(cm):
    """Accuracy + moyennes macro et pondérées de la précision, du rappel et du F1."""
    prec, rap, f1, sup = metriques_depuis_matrice(cm)
    return {
        "accuracy": float(np.trace(cm) / cm.sum()),
        "precision_macro": float(prec.mean()),
        "rappel_macro": float(rap.mean()),
        "f1_macro": float(f1.mean()),
        "precision_pond": float((prec * sup).sum() / sup.sum()),
        "rappel_pond": float((rap * sup).sum() / sup.sum()),
        "f1_pond": float((f1 * sup).sum() / sup.sum()),
    }


def afficher_tableau_global(cms):
    """Tableau récapitulatif. cms = {"Train": cm, "Validation": cm, "Test": cm}"""
    print(f"{'Ensemble':<12}{'Accuracy':>10}{'Préc. macro':>13}{'Rappel macro':>14}{'F1 macro':>10}{'F1 pond.':>10}")
    print("-" * 69)
    for nom, cm in cms.items():
        r = resume_metriques(cm)
        print(f"{nom:<12}{r['accuracy']:>10.4f}{r['precision_macro']:>13.4f}"
              f"{r['rappel_macro']:>14.4f}{r['f1_macro']:>10.4f}{r['f1_pond']:>10.4f}")


def afficher_metriques_par_classe(nom, cm, classes):
    """Précision, rappel et F1 de chaque classe, triés par F1 décroissant."""
    prec, rap, f1, sup = metriques_depuis_matrice(cm)
    print(f"=== {nom} ({cm.sum()} recettes) ===")
    print(f"{'Classe':<15}{'Précision':>10}{'Rappel':>10}{'F1':>10}{'Support':>10}")
    print("-" * 55)
    for k in np.argsort(-f1):
        print(f"{classes[k]:<15}{prec[k]:>10.4f}{rap[k]:>10.4f}{f1[k]:>10.4f}{sup[k]:>10d}")


def afficher_matrice(cm, classes, titre, normaliser=True, fichier=None):
    """
    Affiche la matrice de confusion (lignes = vraie classe, colonnes = classe prédite).
    normaliser=True  : chaque ligne est divisée par son total -> pourcentages (= rappel sur la diagonale)
    normaliser=False : nombres de recettes
    fichier : si renseigné, sauvegarde l'image (utile pour le rapport)
    """
    n = len(classes)
    if normaliser:
        m = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        annot = np.rint(m * 100).astype(int)
    else:
        m = cm
        annot = cm

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(m, cmap="Blues", vmin=0, vmax=1 if normaliser else None)
    ax.set_xticks(range(n))
    ax.set_xticklabels(classes, rotation=90)
    ax.set_yticks(range(n))
    ax.set_yticklabels(classes)
    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe réelle")
    ax.set_title(titre + (" (en % de chaque classe réelle)" if normaliser else ""))

    seuil = m.max() / 2
    for i in range(n):
        for j in range(n):
            if annot[i, j] > 0:
                ax.text(j, i, annot[i, j], ha="center", va="center", fontsize=6,
                        color="white" if m[i, j] > seuil else "black")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    if fichier:
        plt.savefig(fichier, dpi=200, bbox_inches="tight")
    plt.show()

Les fonctions sont écrites pour ne dépendre que de la matrice de confusion : on la calcule une fois par ensemble, puis précision, rappel et F1 en découlent. Elles servent aux deux expériences (sans et avec apprentissage).

Résultats sur le train

Accuracy et précision du modèle sur les données qui ont servi à le construire.

In [ ]:
acc_train, prec_macro_train, prec_pond_train = rapport("TRAIN", train_data, class_vectors, mot_index)

Résultats sur validation et test

In [ ]:
acc_val, prec_macro_val, prec_pond_val = rapport("VALIDATION", val_data, class_vectors, mot_index, detail=False)
print()
acc_test, prec_macro_test, prec_pond_test = rapport("TEST", test_data, class_vectors, mot_index, detail=False)

### Rappel, F1 et matrice de confusion (sans apprentissage)

On calcule la matrice de confusion du modèle sans apprentissage sur le train, la validation et le test, puis on affiche le tableau récapitulatif.

In [ ]:
cms_sans_appr = {}
for nom, dataset in [("Train", train_data), ("Validation", val_data), ("Test", test_data)]:
    y_true, y_pred = predict_dataset(dataset, class_vectors, mot_index)
    cms_sans_appr[nom] = matrice_confusion(y_true, y_pred, classes_list)

afficher_tableau_global(cms_sans_appr)

Détail par classe sur le **test** (l'ensemble qui n'a pas servi à construire le modèle). Pour un autre ensemble, remplacer `"Test"` par `"Train"` ou `"Validation"`.

In [ ]:
afficher_metriques_par_classe("TEST - sans apprentissage", cms_sans_appr["Test"], classes_list)

Matrice de confusion (test). Les valeurs sont normalisées par ligne : chaque case est le pourcentage des recettes d'une vraie cuisine qui ont été prédites dans la cuisine de la colonne. La diagonale correspond donc au rappel de chaque classe. Mettre `normaliser=False` pour avoir les nombres de recettes.

In [ ]:
afficher_matrice(cms_sans_appr["Test"], classes_list,
                 "Matrice de confusion - sans apprentissage (test)",
                 fichier="confusion_sans_apprentissage.png")

# Avec Apprentissage

In [ ]:
classes_list = sorted(classe_counts.keys())
classe_index = {c: i for i, c in enumerate(classes_list)}

def encode(dataset):
    X = np.zeros((len(dataset), len(vocabulary)), dtype=np.float32)
    y = np.zeros(len(dataset), dtype=np.int64)
    for i, recipe in enumerate(dataset):
        for ing in recipe["ingredients"]:
            j = mot_index.get(ing)
            if j is not None:          # on ignore les ingrédients hors vocabulaire
                X[i, j] = 1.0
        y[i] = classe_index[recipe["cuisine"]]
    return torch.from_numpy(X), torch.from_numpy(y)

X_train, y_train = encode(train_data)
X_val, y_val = encode(val_data)
X_test, y_test = encode(test_data)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

Creation du Neural Network avec la couche d'entrée de la taille du vocabulaire, 2 couche caché de 128 et 64 neuronne et la couche de sortie avec 20 neurone pour chaque classe sur lesquelles nous appliquons un softmax pour avoir un classe de sortie.

Le choix de la taille des 2 couches caché à été demandé à Chat GPT

In [ ]:
class CuisineNet(nn.Module):
    def __init__(self, n_in, h1=128, h2=64, n_out=20, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h2, n_out),
        )

    def forward(self, x):
        return self.net(x)

    def predict_proba(self, x):
        return torch.softmax(self.forward(x), dim=1)


model = CuisineNet(n_in=len(vocabulary), n_out=len(classes_list))
print(model)

Choix des parametres

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Entrainement

In [ ]:
# code generer par chat gpt
import copy

EPOCHS = 30

best_val_loss = float("inf")
best_state = None
epochs_sans_amelioration = 0

for epoch in range(1, EPOCHS + 1):
    # --- entraînement
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
    train_loss /= len(train_loader.dataset)

    # --- validation
    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            out = model(xb)
            val_loss += criterion(out, yb).item() * len(xb)
            correct += (out.argmax(dim=1) == yb).sum().item()
    val_loss /= len(val_loader.dataset)
    val_acc = correct / len(val_loader.dataset)

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.4f}")

    # --- early stopping : on garde les meilleurs poids
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

Analyse des resultats

In [ ]:
def rapport(nom, X, y, detail=True):
    model.eval()
    with torch.no_grad():
        y_pred = model.predict_proba(X).argmax(dim=1).numpy()
    y_true = y.numpy()

    acc = float(np.mean(y_pred == y_true))
    prec, supports = [], []
    for k in range(len(classes_list)):
        predits_k = np.sum(y_pred == k)
        vp = np.sum((y_pred == k) & (y_true == k))
        prec.append(vp / predits_k if predits_k > 0 else 0.0)
        supports.append(np.sum(y_true == k))
    prec, supports = np.array(prec), np.array(supports)

    print(f"=== {nom} ({len(y_true)} recettes) ===")
    print(f"Accuracy             : {acc:.4f}")
    print(f"Précision (macro)    : {prec.mean():.4f}")
    print(f"Précision (pondérée) : {(prec * supports).sum() / supports.sum():.4f}")
    if detail:
        print("\nPrécision par classe :")
        for k in np.argsort(-prec):
            print(f"  {classes_list[k]:<14} {prec[k]:.4f}  (support : {supports[k]})")

rapport("TRAIN", X_train, y_train)
print()
rapport("VALIDATION", X_val, y_val, detail=False)
print()
rapport("TEST", X_test, y_test, detail=False)

### Rappel, F1 et matrice de confusion (avec apprentissage)

Même démarche avec le réseau de neurones. On récupère les classes prédites par le modèle, puis on calcule la matrice de confusion de chaque ensemble.

In [ ]:
def predire_nn(X):
    """Renvoie la cuisine prédite (nom) pour chaque ligne de X."""
    model.eval()
    with torch.no_grad():
        indices = model.predict_proba(X).argmax(dim=1).tolist()
    return [classes_list[i] for i in indices]


cms_nn = {}
for nom, X, y in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    y_true = [classes_list[i] for i in y.tolist()]
    cms_nn[nom] = matrice_confusion(y_true, predire_nn(X), classes_list)

afficher_tableau_global(cms_nn)

In [ ]:
afficher_metriques_par_classe("TEST - avec apprentissage", cms_nn["Test"], classes_list)

In [ ]:
afficher_matrice(cms_nn["Test"], classes_list,
                 "Matrice de confusion - avec apprentissage (test)",
                 fichier="confusion_avec_apprentissage.png")

### Comparaison des deux expériences (test)

In [ ]:
print(f"{'Modèle (test)':<26}{'Accuracy':>10}{'Précision':>11}{'Rappel':>9}{'F1':>9}")
print("-" * 65)
for nom, cms in [("Sans apprentissage", cms_sans_appr), ("Avec apprentissage (NN)", cms_nn)]:
    r = resume_metriques(cms["Test"])
    print(f"{nom:<26}{r['accuracy']:>10.4f}{r['precision_macro']:>11.4f}{r['rappel_macro']:>9.4f}{r['f1_macro']:>9.4f}")
print("\n(Précision, rappel et F1 : moyennes macro sur les 20 classes)")

In [ ]:
# fait par chat gpt
def afficher_erreurs(nom, dataset, X, y, n=30, mode="confiantes", seed=42):
    """
    Affiche n recettes mal classées.
    mode = "confiantes" : les erreurs où le modèle était le plus sûr de lui (les plus instructives)
    mode = "aleatoires" : des erreurs tirées au hasard
    """
    model.eval()
    with torch.no_grad():
        probas = model.predict_proba(X)
    y_pred = probas.argmax(dim=1)

    erreurs = (y_pred != y).nonzero(as_tuple=True)[0].numpy()
    print(f"=== {nom} : {len(erreurs)} erreurs sur {len(y)} recettes ({len(erreurs) / len(y) * 100:.1f}%) ===\n")

    if mode == "confiantes":
        confiance = probas[erreurs, y_pred[erreurs]].numpy()
        choisis = erreurs[np.argsort(-confiance)[:n]]
    else:
        choisis = np.random.default_rng(seed).choice(erreurs, size=min(n, len(erreurs)), replace=False)

    for i in choisis:
        vraie = classes_list[y[i].item()]
        predite = classes_list[y_pred[i].item()]
        print(f"Recette #{i}")
        print(f"  Vraie cuisine   : {vraie}  (proba donnée par le modèle : {probas[i, y[i]].item():.3f})")
        print(f"  Cuisine prédite : {predite}  (proba : {probas[i, y_pred[i]].item():.3f})")
        print(f"  Ingrédients     : {', '.join(dataset[i]['ingredients'])}")
        print()

afficher_erreurs("VALIDATION", val_data, X_val, y_val, n=25, mode="confiantes")

### Le choix de 5 erreurs avec les explications

Toutes les erreurs que nous avons trouver sur le modèle sont la conséquence de la différence de proportion des differents types de cuisine dans le data, il  est alors plus entrainé a trouver correctement une cuisine italienne ou mexicaine qui est en majorité dans le dataset.

#### Erreur 1
Recette #1087
  Vraie cuisine   : indian  (proba donnée par le modèle : 0.038)
  Cuisine prédite : mexican  (proba : 0.892)
  Ingrédients     : tomatoes, salt, chili powder, canola oil, garlic, tamarind extract, onions

Ici on voit tres clairement le probleme de disproportion des classes dans le dataset avec la cuisine mexicaine à 16% tandis que la cuisine indienne est à 7.5%

#### Erreur 2
Recette #4489
  Vraie cuisine   : thai  (proba donnée par le modèle : 0.057)
  Cuisine prédite : mexican  (proba : 0.886)
  Ingrédients     : lime juice, Thai fish sauce, brown sugar, grapefruit, avocado, purple onion, chopped cilantro fresh, red chili peppers, uncook medium shrimp, peel and devein

Pour cette erreur on voit tres clairement le probleme de la majorité de cuisine mexicaine dans le dataset, mais d'autre part en augmentant le vocabulaire on voit que Thai fish sauce aurait pus apparaitre dedans et jouer alors un jeu decisif dans le choix

### Erreur 3
Recette #4526
  Vraie cuisine   : vietnamese  (proba donnée par le modèle : 0.126)
  Cuisine prédite : thai  (proba : 0.871)
  Ingrédients     : groundnut, lime juice, spring onions, rice flour, ground turmeric, fish sauce, fresh coriander, vegetables, garlic cloves, coconut milk, caster sugar, shiitake, sea salt, beansprouts, red chili peppers, water, lettuce leaves, king prawns, onions

Ici dans cette erreur il ne s'agit pas de proportion dans le dataset mais de similitude des cuisines


### Erreur 4
Recette #5752
  Vraie cuisine   : korean  (proba donnée par le modèle : 0.046)
  Cuisine prédite : chinese  (proba : 0.884)
  Ingrédients     : water, sesame oil, sugar, light soy sauce, beef tenderloin, dark soy sauce, minced ginger, crushed garlic, black pepper, green onions, white sesame seeds

Malgré la différence la proportion entre la cuisine chinoise et coreene, ces deux cuisine sont similaire car venant de la meme region et l'on peut en deduire que le modele trouve la bonne region

### Erreur 5
Recette #5896
  Vraie cuisine   : russian  (proba donnée par le modèle : 0.001)
  Cuisine prédite : southern_us  (proba : 0.912)
  Ingrédients     : sugar, salt, baking powder, oil, large eggs, all-purpose flour, buttermilk

Ici il faut regarder les ingredients et les ingredients sont present dans toutes les cuisine, donc le modele a choisit par probabilité une des classe les plus présentes.

# Classification de cuisines (What's Cooking) avec un Transformer pré-entraîné

**Tâche** : prédire la cuisine (20 classes) d'une recette à partir de sa liste d'ingrédients (texte anglais).

**Choix documentés**

| Élément | Choix |
|---|---|
| Modèle | `roberta-base` (125 M paramètres, pré-entraîné sur de l'anglais) |
| Tête | `RobertaClassificationHead` (dense 768→768, tanh, dropout, projection 768→20) sur le token `<s>` |
| Tokenisation | BPE byte-level de RoBERTa, texte = ingrédients en minuscules séparés par `", "` |
| Longueur max | déterminée sur les données (percentile 99,9 des longueurs en tokens, plafonnée à 128) |
| Padding | dynamique par batch (`DataCollatorWithPadding`) |
| Phase 1 | encodeur gelé, entraînement de la tête seule (lr 1e-3) |
| Phase 2 | fine-tuning complet (lr 2e-5, warmup 6 %, weight decay 0,01) |
| Validation | 10 % du train, split stratifié (seed 42) ; métrique de sélection : macro-F1 |

In [ ]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn

In [ ]:
import json, os, random, time
from google.colab import files
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "ATTENTION : pas de GPU (Runtime > Modifier le type d'exécution > GPU)")

# --- Chemins : uploadez train.json et test.json dans /content (ou montez Drive) ---
DATA_DIR = "/content"
# from google.colab import drive; drive.mount('/content/drive'); DATA_DIR = "/content/drive/MyDrive/data"
OUT_DIR = "/content/outputs"; os.makedirs(OUT_DIR, exist_ok=True)

## 1. Données

In [ ]:
uploaded = files.upload()   # une fenêtre s'ouvre : sélectionnez train.json ET test.json
print(list(uploaded.keys()))

In [ ]:
train_raw = json.load(open(f"/content/train.json"))
test_raw  = json.load(open(f"/content/test.json"))

def to_text(ingredients):
    # minuscules, espaces normalisés, séparateur virgule : les ingrédients multi-mots restent lisibles
    return ", ".join(" ".join(i.lower().split()) for i in ingredients)

df = pd.DataFrame({"id":[r["id"] for r in train_raw],
                   "text":[to_text(r["ingredients"]) for r in train_raw],
                   "cuisine":[r["cuisine"] for r in train_raw]})
df_test = pd.DataFrame({"id":[r["id"] for r in test_raw],
                        "text":[to_text(r["ingredients"]) for r in test_raw]})

n0 = len(df)
df = df.drop_duplicates(subset=["text","cuisine"]).reset_index(drop=True)   # évite des doublons exacts entre train et validation
print(f"train: {n0} -> {len(df)} après dédoublonnage | test: {len(df_test)}")

labels = sorted(df["cuisine"].unique())
label2id = {l:i for i,l in enumerate(labels)}; id2label = {i:l for l,i in label2id.items()}
df["label"] = df["cuisine"].map(label2id)
print(len(labels), "classes\n", df["cuisine"].value_counts(normalize=True).round(3).to_string())

tr_df, va_df = train_test_split(df, test_size=0.10, stratify=df["label"], random_state=SEED)
print("train/val:", len(tr_df), len(va_df))

## 2. Modèle et tokenisation
`roberta-base` est un modèle anglais ; le corpus est en anglais (noms d'ingrédients). Son BPE byte-level gère bien les mots rares/fautifs (« mayonaise »).

In [ ]:
MODEL_NAME = "roberta-base"                       # modèle pré-entraîné à charger
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # tokenizer associé au modèle

# Longueur max choisie d'après la distribution réelle (train + test)
all_texts = pd.concat([df["text"], df_test["text"]]).tolist()  # tous les textes en une liste
# nombre de tokens par texte (avec <s> et </s>), stocké dans un tableau NumPy
lens = np.array([len(x) for x in tokenizer(all_texts, add_special_tokens=True)["input_ids"]])
for p in (50, 95, 99, 99.9, 100):                 # percentiles à examiner
    print(f"percentile {p:>5}: {np.percentile(lens, p):.0f} tokens")  # longueur sous laquelle se trouvent p % des textes
MAX_LEN = int(min(128, np.ceil(np.percentile(lens, 99.9))))  # couvre 99,9 % des textes, plafonné à 128
print(f"MAX_LEN retenu = {MAX_LEN} | part tronquée: {(lens > MAX_LEN).mean():.4%}")  # valeur retenue et % de textes tronqués

In [ ]:
def make_ds(d):
    ds = Dataset.from_pandas(d[["text","label"]].reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

ds_tr, ds_va = make_ds(tr_df), make_ds(va_df)
ds_te = Dataset.from_pandas(df_test[["text"]]).map(
    lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True, remove_columns=["text"])
collator = DataCollatorWithPadding(tokenizer)
print(tokenizer.convert_ids_to_tokens(ds_tr[0]["input_ids"])[:25])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id)
n_total = sum(p.numel() for p in model.parameters())
n_head  = sum(p.numel() for p in model.classifier.parameters())
print(f"paramètres: {n_total/1e6:.1f} M | tête de classification: {n_head/1e6:.2f} M")

def compute_metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro")}

fp16 = device == "cuda"
bf16_ok = fp16 and torch.cuda.is_bf16_supported()

## 3. Phase 1 — entraînement de la tête seule (encodeur gelé)
Les représentations de RoBERTa sont figées ; seul `model.classifier` est appris, avec un learning rate élevé.

In [ ]:
for p in model.roberta.parameters(): p.requires_grad = False
print("paramètres entraînables:", sum(p.numel() for p in model.parameters() if p.requires_grad))

P1 = dict(epochs=4, lr=1e-3, bs=64, wd=0.01, warmup=0.06)

# Calculate warmup_steps from warmup_ratio
total_train_samples = len(ds_tr)
train_batch_size = P1["bs"]
num_train_epochs = P1["epochs"]
warmup_ratio = P1["warmup"]

num_update_steps_per_epoch = (total_train_samples + train_batch_size - 1) // train_batch_size
max_train_steps = num_train_epochs * num_update_steps_per_epoch
num_warmup_steps = int(max_train_steps * warmup_ratio)

args1 = TrainingArguments(
    output_dir=f"{OUT_DIR}/phase1", num_train_epochs=P1["epochs"], learning_rate=P1["lr"],
    per_device_train_batch_size=P1["bs"], per_device_eval_batch_size=128,
    weight_decay=P1["wd"], warmup_steps=num_warmup_steps, lr_scheduler_type="linear", # Changed warmup_ratio to warmup_steps
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    fp16=fp16 and not bf16_ok, bf16=bf16_ok, logging_steps=100, report_to="none", seed=SEED)
t1 = Trainer(model=model, args=args1, train_dataset=ds_tr, eval_dataset=ds_va,
             data_collator=collator, compute_metrics=compute_metrics)
t0 = time.time(); t1.train(); print(f"durée phase 1: {time.time()-t0:.0f}s")
m1 = t1.evaluate(); print("Validation après phase 1 (tête seule):", m1)

## 4. Phase 2 — fine-tuning complet
On dégèle tout l'encodeur avec un learning rate faible pour ne pas détruire le pré-entraînement.

In [ ]:
for p in model.parameters(): p.requires_grad = True

P2 = dict(epochs=4, lr=2e-5, bs=32, wd=0.01, warmup=0.06)

# Calculate warmup_steps from warmup_ratio for Phase 2
total_train_samples = len(ds_tr)
train_batch_size_p2 = P2["bs"]
num_train_epochs_p2 = P2["epochs"]
warmup_ratio_p2 = P2["warmup"]

num_update_steps_per_epoch_p2 = (total_train_samples + train_batch_size_p2 - 1) // train_batch_size_p2
max_train_steps_p2 = num_train_epochs_p2 * num_update_steps_per_epoch_p2
num_warmup_steps_p2 = int(max_train_steps_p2 * warmup_ratio_p2)

args2 = TrainingArguments(
    output_dir=f"{OUT_DIR}/phase2", num_train_epochs=P2["epochs"], learning_rate=P2["lr"],
    per_device_train_batch_size=P2["bs"], per_device_eval_batch_size=128,
    weight_decay=P2["wd"], warmup_steps=num_warmup_steps_p2, lr_scheduler_type="linear", # Changed warmup_ratio to warmup_steps
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    fp16=fp16 and not bf16_ok, bf16=bf16_ok, logging_steps=100, report_to="none", seed=SEED)
t2 = Trainer(model=model, args=args2, train_dataset=ds_tr, eval_dataset=ds_va,
             data_collator=collator, compute_metrics=compute_metrics)
t0 = time.time(); t2.train(); print(f"durée phase 2: {time.time()-t0:.0f}s")
m2 = t2.evaluate(); print("Validation après phase 2 (fine-tuning complet):", m2)

## 5. Évaluation détaillée

In [ ]:
pred_va = t2.predict(ds_va)
y_true, y_pred = pred_va.label_ids, pred_va.predictions.argmax(-1)
print(classification_report(y_true, y_pred, target_names=labels, digits=3))

cm = confusion_matrix(y_true, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)

# Valeur normalisée dans chaque case
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > 0.5 else "black")

ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
ax.set_xlabel("prédit"); ax.set_ylabel("réel")
ax.set_title("Matrice de confusion normalisée (validation)")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=150)
plt.show()

## 6. Prédictions sur `test.json`, sauvegarde et documentation

In [ ]:
pred_te = t2.predict(ds_te).predictions.argmax(-1)
sub = pd.DataFrame({"id": df_test["id"], "cuisine": [id2label[i] for i in pred_te]})
sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)
print(sub.head(), "\n", sub["cuisine"].value_counts(normalize=True).round(3).head(6).to_string())

model.save_pretrained(f"{OUT_DIR}/model"); tokenizer.save_pretrained(f"{OUT_DIR}/model")

card = {
  "model": MODEL_NAME, "n_parameters": n_total, "n_head_parameters": n_head,
  "head": "RobertaClassificationHead (dense 768->768, tanh, dropout 0.1, out_proj 768->%d)" % len(labels),
  "tokenizer": {"type": "byte-level BPE (RoBERTa)", "text_format": "ingrédients en minuscules joints par ', '",
                "max_length": MAX_LEN, "truncation": True, "padding": "dynamique par batch",
                "truncated_share": float((lens > MAX_LEN).mean())},
  "data": {"train_after_dedup": len(df), "train": len(tr_df), "val": len(va_df), "test": len(df_test),
           "classes": labels, "split": "stratifié 90/10, seed 42"},
  "phase1_head_only": {**P1, "frozen": "roberta encoder", "scheduler": "linear", "val": m1},
  "phase2_full_finetune": {**P2, "scheduler": "linear", "val": m2},
  "optimizer": "AdamW (défaut Trainer, betas 0.9/0.999, eps 1e-8)",
  "precision": "bf16" if bf16_ok else ("fp16" if fp16 else "fp32"),
  "loss": "cross-entropy", "selection_metric": "macro_f1 sur validation (meilleure epoch)", "seed": SEED,
}
json.dump(card, open(f"{OUT_DIR}/model_card.json", "w"), indent=2, ensure_ascii=False, default=float)
print(json.dumps(card, indent=2, ensure_ascii=False, default=float))